# Anàlisi general taules de dades csv PADRIS

In [2]:
import pandas as pd
import numpy as np
import random as rd

## 1. Taula 5 Derivació dels pacients

In [ ]:
path = r'C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\689_F_ap_derivacio_sm_Final.csv'
df_derivacio = pd.read_csv(path)

In [ ]:
df_derivacio.head()

,id,idCas,Identificador de registre CMBD,Unitat Proveïdora,UP_descripcio,Data inici assistència,Tipus de professional,Tipus_professional_descrip,Unitat Proveïdora Destí,Tipus UP Destí
0,1,83892072,263736108,337,EAP Baix Berguedà,3/27/2015,1,1 Metge/ssa de família,CSM Adults Berguedà,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
1,2,83892237,306747946,1933,EAP Barcelona 7B - Sardenya,10/21/2015,1,1 Metge/ssa de família,CSM Adults Guinardó,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
2,3,83897885,250764528,294,EAP Badalona 3 - Progrés-Raval,1/8/2015,1,1 Metge/ssa de família,CSM Adults Badalona 1 Est,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
3,4,83900256,278190986,157,EAP Garraf Rural,7/9/2015,1,1 Metge/ssa de família,CSM Garraf,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA
4,5,83902712,257236651,1798,EAP Palamós,2/19/2015,2,2 Pediatre/a,CSM Adults Baix Empordà,ASSISTÈNCIA SALUT MENTAL - AMBULATÒRIA


In [ ]:
df_derivacio.Tipus_professional_descrip.unique()

array(['1 Metge/ssa de família', '2 Pediatre/a', '4 Infermer/a',
       '3 Odontòleg/oga', '5 Treballador/a social'], dtype=object)

In [ ]:
df_derivacio["Tipus de professional"].unique()

array([1, 2, 4, 3, 5])

In [ ]:
df_derivacio.id.count()

np.int64(7195)

## 2. Taula sociodemogràfica: taula 3, tabac, IMC

VAL:
- 0= No fumador 
- 1= Fumador 
- 2 = exfumador

In [8]:
df_tabac=pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\8_tabac_cohort_total_Final.csv")
df_tabac.head()

,id,idCas,DAT,DBAIXA,VAL
0,1,73405423,2012-11-12T22:00:00.000Z,2017-12-30T22:00:00.000Z,0
1,2,73405425,2016-06-08T20:00:00.000Z,2017-12-30T22:00:00.000Z,0
2,3,73405429,2016-10-23T20:00:00.000Z,2017-12-30T22:00:00.000Z,0
3,4,73405436,2017-03-05T22:00:00.000Z,2017-12-30T22:00:00.000Z,0
4,5,78465847,2017-11-07T22:00:00.000Z,2017-12-30T22:00:00.000Z,0


In [9]:
df_tabac.VAL.unique()

array([0, 2, 1])

In [37]:
# Filtering rows for just patients inside the study cohort
patients_study = pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\3_taula_0_cohort_estudi_Final.csv")

# Filter df to only keep rows where idCas is in the patients file
df_tabac_study = df_tabac[df_tabac["idCas"].isin(patients_study["idCas"])].copy()

print(f"Original rows:  {len(df_tabac)}")
print(f"Filtered rows:  {len(df_tabac_study)}")

C:\Users\laiar\AppData\Local\Temp\ipykernel_13928\203110669.py:2: DtypeWarning: Columns (7,9,11) have mixed types. Specify dtype option on import or set low_memory=False.
  patients_study = pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\3_taula_0_cohort_estudi_Final.csv")


Original rows:  1062615
Filtered rows:  419825


In [ ]:
# Filtering for schizophrenia patients

To check if VAL is correct, I need to understand the logic:

- 0 = never smoked → DAT should be null/empty, DBAIXA should be null/empty
- 1 = currently smokes → DAT has a date, DBAIXA is null/empty
- 2 = ex-smoker → DAT has a date, DBAIXA also has a date

In [17]:
nan_dbaixa = df_tabac_study[df_tabac_study["DBAIXA"].isna()].copy()
print(f"Rows with NaN in DBAIXA: {len(nan_dbaixa)}")
print(nan_dbaixa)

Rows with NaN in DBAIXA: 0
Empty DataFrame
Columns: [id, idCas, DAT, DBAIXA, VAL]
Index: []


In [18]:
# Convert dates — no warning should appear
df_tabac_study["DAT"] = pd.to_datetime(df_tabac_study["DAT"], utc=True)
df_tabac_study["DBAIXA"] = pd.to_datetime(df_tabac_study["DBAIXA"], utc=True)

In [19]:
placeholder = pd.Timestamp("2017-12-30 22:00:00", tz="UTC")

# Validation function
def expected_val(row):
    has_real_dbaixa = pd.notna(row["DBAIXA"]) and row["DBAIXA"] != placeholder

    if has_real_dbaixa:
        return 2  # Has a real stop date → ex-smoker
    else:
        return 1  # No real stop date → currently smokes

df_tabac_study["VAL_expected"] = df_tabac_study.apply(expected_val, axis=1)

# Flag mismatches (skip VAL=0 as it's unverifiable from dates alone)
df_tabac_study["VAL_correct"] = df_tabac_study.apply(lambda row:
    True if row["VAL"] == 0
    else row["VAL"] == row["VAL_expected"], axis=1
)

# Summary
total = len(df_tabac_study)
correct = df_tabac_study["VAL_correct"].sum()
incorrect = total - correct

print(f"Total rows:            {total}")
print(f"Correct/Unverifiable:  {correct}")
print(f"Clear errors:          {incorrect}")
print()

# Error breakdown
print("Error breakdown:")
print(f"  VAL=1 but has real DBAIXA (should be 2): {((df_tabac_study['VAL']==1) & (df_tabac_study['DBAIXA'] != placeholder)).sum()}")
print(f"  VAL=2 but DBAIXA is placeholder (should be 1): {((df_tabac_study['VAL']==2) & (df_tabac_study['DBAIXA'] == placeholder)).sum()}")
print()

# VAL=0 anomalies
print("VAL=0 anomalies (suspicious):")
print(f"  VAL=0 with real DBAIXA (looks like ex-smoker): {((df_tabac_study['VAL']==0) & (df_tabac_study['DBAIXA'] != placeholder)).sum()}")
print(f"  VAL=0 with placeholder DBAIXA (looks like smoker): {((df_tabac_study['VAL']==0) & (df_tabac_study['DBAIXA'] == placeholder)).sum()}")

Total rows:            419825
Correct/Unverifiable:  314365
Clear errors:          105460

Error breakdown:
  VAL=1 but has real DBAIXA (should be 2): 64710
  VAL=2 but DBAIXA is placeholder (should be 1): 40750

VAL=0 anomalies (suspicious):
  VAL=0 with real DBAIXA (looks like ex-smoker): 23287
  VAL=0 with placeholder DBAIXA (looks like smoker): 149551


### Canvi dades "VAL" a partir de les dates de inici i final

In [20]:
# Guarda el VAL original abans de canviar res
df_tabac_study["VAL_original"] = df_tabac_study["VAL"]

# Corregeix VAL basat en les dates
df_tabac_study.loc[(df_tabac_study["VAL"] == 1) & (df_tabac_study["DBAIXA"] != placeholder), "VAL"] = 2
df_tabac_study.loc[(df_tabac_study["VAL"] == 2) & (df_tabac_study["DBAIXA"] == placeholder), "VAL"] = 1

print("VAL corregit. Nova distribució:")
print(df_tabac_study["VAL"].value_counts())

VAL corregit. Nova distribució:
VAL
0    172838
1    140840
2    106147
Name: count, dtype: int64


In [21]:
df_tabac_study.head()

,id,idCas,DAT,DBAIXA,VAL,VAL_expected,VAL_correct,VAL_original
0,1,73405423,2012-11-12 22:00:00+00:00,2017-12-30 22:00:00+00:00,0,1,True,0
1,2,73405425,2016-06-08 20:00:00+00:00,2017-12-30 22:00:00+00:00,0,1,True,0
2,3,73405429,2016-10-23 20:00:00+00:00,2017-12-30 22:00:00+00:00,0,1,True,0
7,8,83886129,2014-07-03 20:00:00+00:00,2017-12-30 22:00:00+00:00,0,1,True,0
9,10,83886183,2012-03-12 22:00:00+00:00,2017-03-23 22:00:00+00:00,2,2,True,2


Check if a patient has multiple rows in the same dataset

In [22]:
# Check if any idCas has multiple rows in df_tabac_study
duplicates = df_tabac_study.groupby("idCas").size()

print(f"Total unique patients: {len(duplicates)}")
print(f"Patients with 1 row: {(duplicates == 1).sum()}")
print(f"Patients with multiple rows: {(duplicates > 1).sum()}")
print()
print("Distribution of rows per patient:")
print(duplicates.value_counts().sort_index())

Total unique patients: 291564
Patients with 1 row: 219844
Patients with multiple rows: 71720

Distribution of rows per patient:
1     219844
2      37812
3      20318
4       8160
5       3287
6       1329
7        469
8        192
9         73
10        40
11        23
12         6
13         5
14         3
15         2
16         1
Name: count, dtype: int64


Merging Smoking status with table 3 that's the Sociodemographic table + deleting duplicate patients by newest date

In [ ]:
# We decide to keep the patient data that's most recently updated (latest DAT) for each patient, as it should reflect the most current smoking status. This is a common approach when dealing with multiple records per patient in longitudinal datasets.

df_tabac_latest = df_tabac_study.sort_values("DAT", ascending=False).drop_duplicates(subset="idCas", keep="first")

print(f"Rows before dedup: {len(df_tabac_study)}")
print(f"Rows after dedup:  {len(df_tabac_latest)}")
print(f"Unique patients:   {df_tabac_latest['idCas'].nunique()}")
print()
print("VAL distribution after dedup:")
print(df_tabac_latest["VAL"].value_counts())

# Now merge safely
cohort = patients_study.merge(
    df_tabac_latest[["idCas", "VAL"]],
    on="idCas",
    how="left"
)

print(f"\nCohort rows after merge: {len(cohort)}")
print(f"VAL null after merge:    {cohort['VAL'].isna().sum()}")

Rows before dedup: 419825
Rows after dedup:  291564
Unique patients:   291564

VAL distribution after dedup:
VAL
0    149701
1    140598
2      1265
Name: count, dtype: int64

Cohort rows after merge: 473812
VAL null after merge:    182248


There are multiple IMC values meaning it has been measured over the years so for the Sociodemographic dataset I'm just adding the first and last value to maybe end up relating it to adherence.

In [26]:
# Now we also merge IMC
# Load IMC data
df_bmi = pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\6_imc_2020_Final.csv")
# Convert DATE to datetime
df_bmi["DATA"] = pd.to_datetime(df_bmi["DATA"], utc=True)

# Filter only patients in cohort
df_bmi_study = df_bmi[df_bmi["idCas"].isin(cohort["idCas"])].copy()

# First BMI per patient
df_bmi_first = df_bmi_study.sort_values("DATA", ascending=True).drop_duplicates(subset="idCas", keep="first")[["idCas", "VALOR"]].rename(columns={"VALOR": "BMI_first"})

# Last BMI per patient
df_bmi_last = df_bmi_study.sort_values("DATA", ascending=False).drop_duplicates(subset="idCas", keep="first")[["idCas", "VALOR"]].rename(columns={"VALOR": "BMI_last"})

# Merge both into cohort
cohort = cohort.merge(df_bmi_first, on="idCas", how="left")
cohort = cohort.merge(df_bmi_last, on="idCas", how="left")

print(f"Cohort rows: {len(cohort)}")
print(f"BMI_first null: {cohort['BMI_first'].isna().sum()}")
print(f"BMI_last null:  {cohort['BMI_last'].isna().sum()}")
print()
print(cohort.head(10))


Cohort rows: 473812
BMI_first null: 415997
BMI_last null:  415985

   id     idCas  SEXE1 SEXE2 EDAT_rang NACIONALITAT1 NACIONALITAT2  \
0   1  67170049      0  Home   20 - 24           752        SUÈCIA   
1   2  67255096      1  Dona   15 - 19           208     DINAMARCA   
2   3  67355661      0  Home   25 - 29           642       ROMANIA   
3   4  67495562      1  Dona   30 - 34           170      COLòMBIA   
4   5  67495914      1  Dona   65 - 69           724       ESPANYA   
5   6  67510399      0  Home   20 - 24           270        GÀMBIA   
6   7  67605922      0  Home   30 - 34           686       SENEGAL   
7   8  67696247      0  Home   50 - 54           724       ESPANYA   
8   9  67930422      1  Dona   10 - 14           170      COLòMBIA   
9  10  67960684      1  Dona     5 - 9           340      HONDURES   

     DATAMORT               NIVSOC  VAL  BMI_first  BMI_last  
0  09/09/9999          2. < 18.000  NaN        NaN       NaN  
1  09/09/9999  3. 18.001 - 100.000  

In [47]:
# BMI coverage
print(f"BMI_first coverage: {(cohort['BMI_first'].notna().sum() / len(cohort)) * 100:.1f}%")
print(f"BMI_last coverage:  {(cohort['BMI_last'].notna().sum() / len(cohort)) * 100:.1f}%")

BMI_first coverage: 12.2%
BMI_last coverage:  12.2%


In [29]:
print("Unique values in VAL:")
print(cohort["VAL"].unique())
print(cohort["VAL"].value_counts())

Unique values in VAL:
[nan  0.  1.  2.]
VAL
0.0    149701
1.0    140598
2.0      1265
Name: count, dtype: int64


In [30]:
print(f"Patients with no smoking record (NaN): {cohort['VAL'].isna().sum()}")
print(f"Total cohort rows: {len(cohort)}")

Patients with no smoking record (NaN): 182248
Total cohort rows: 473812


In [32]:
print(f"Total cohort:              {len(cohort)}")
print(f"With smoking record:       {cohort['VAL'].notna().sum()}")
print(f"Without smoking record:    {cohort['VAL'].isna().sum()}")
print(f"% missing:                 {cohort['VAL'].isna().mean()*100:.1f}%")

# Check if these patients were in the original smoking table at all
missing_smoking = cohort[cohort["VAL"].isna()]["idCas"]
in_tabac = missing_smoking.isin(df_tabac_study["idCas"])
print(f"\nOf the missing patients:")
print(f"  In original smoking table but lost in merge: {in_tabac.sum()}")
print(f"  Never in smoking table at all:               {(~in_tabac).sum()}")

Total cohort:              473812
With smoking record:       291564
Without smoking record:    182248
% missing:                 38.5%

Of the missing patients:
  In original smoking table but lost in merge: 0
  Never in smoking table at all:               182248


In [ ]:
cols_to_drop = ["ABS1", "ABS2", "SS1", "SS2", "RS1", "RS2", "SITASS1", "SITASS2", "INDFAR1", "INDFAR2", "SUBINDFAR1", "SUBINDFAR2"]
cohort = cohort.drop(columns=cols_to_drop)
print(f"Remaining columns ({len(cohort.columns)}):")
print(cohort.columns.tolist())

Remaining columns (12):
['id', 'idCas', 'SEXE1', 'SEXE2', 'EDAT_rang', 'NACIONALITAT1', 'NACIONALITAT2', 'DATAMORT', 'NIVSOC', 'VAL', 'BMI_first', 'BMI_last']


We checked that the patients with Nan values are the ones that are not even in the smoking csv.

Now we want to turn VAL data into encoded data with OneHotEncoder

In [44]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# OHE only on non-null VAL rows
ohe = OneHotEncoder(sparse_output=False)

non_null_mask = cohort["VAL"].notna()
val_encoded = ohe.fit_transform(cohort.loc[non_null_mask, ["VAL"]])

val_df = pd.DataFrame(val_encoded, columns=["never_smoker", "current_smoker", "ex_smoker"], index=cohort[non_null_mask].index)

# Add to cohort (NaN rows will have NaN in the new columns)
cohort = pd.concat([cohort, val_df], axis=1)

print(f"Rows with NaN in OHE columns: {cohort['never_smoker'].isna().sum()}")

# Save
cohort.to_csv(r"C:\Users\laiar\Desktop\TFG\data_sintetica\Data-sintetica\src\csv merged\cohort_estudi_sociodemografica.csv", index=False)

Rows with NaN in OHE columns: 182248


Now we drop columns we don't need from cohort study 3 and generate a new csv that's where we're gonna have the Sociodemographic data

In [38]:
# Drop duplicate OHE columns if they exist
cols_to_check = ["never_smoker", "current_smoker", "ex_smoker", "has_smoking"]
cohort = cohort.loc[:, ~cohort.columns.duplicated()]

# Also drop has_smoking if it was added
cohort = cohort.drop(columns=[c for c in cols_to_check if c in cohort.columns], errors="ignore")

print(f"Columns after cleanup: {len(cohort.columns)}")
print(cohort.columns.tolist())

Columns after cleanup: 12
['id', 'idCas', 'SEXE1', 'SEXE2', 'EDAT_rang', 'NACIONALITAT1', 'NACIONALITAT2', 'DATAMORT', 'NIVSOC', 'VAL', 'BMI_first', 'BMI_last']


In [46]:
print(cohort.columns.tolist())
print(f"\nTotal rows: {len(cohort)}")
print(f"Total columns: {len(cohort.columns)}")
print(cohort.dtypes)

['id', 'idCas', 'SEXE1', 'SEXE2', 'EDAT_rang', 'NACIONALITAT1', 'NACIONALITAT2', 'DATAMORT', 'NIVSOC', 'VAL', 'BMI_first', 'BMI_last', 'never_smoker', 'current_smoker', 'ex_smoker']

Total rows: 473812
Total columns: 15
id                  int64
idCas               int64
SEXE1               int64
SEXE2              object
EDAT_rang          object
NACIONALITAT1      object
NACIONALITAT2      object
DATAMORT           object
NIVSOC             object
VAL               float64
BMI_first         float64
BMI_last          float64
never_smoker      float64
current_smoker    float64
ex_smoker         float64
dtype: object


## Cohort study, patients who ever had a Schizophrenia diagnosis

In [50]:
df_dx=pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\11_ap_dx_sm_cohort_Final.csv")

# F20 is the code CIE-10 for schizophrenia
df_schiz = df_dx[df_dx["PR_COD_PS"].str.contains("F20", na=False)]

print(f"Rows with schizophrenia diagnosis: {len(df_schiz)}")
print(f"Unique patients: {df_schiz['idCas'].nunique()}")
print()
print("F20 subtypes found:")
print(df_schiz[["PR_COD_PS", "PS_DES"]].drop_duplicates().sort_values("PR_COD_PS").to_string(index=False))

# Get all unique patients with any F20 diagnosis
schiz_patients = df_schiz["idCas"].unique()

# Filter cohort to schizophrenia patients only
cohort_schiz = cohort[cohort["idCas"].isin(schiz_patients)].copy()

print(f"Total cohort:           {len(cohort)}")
print(f"Schizophrenia patients: {len(cohort_schiz)}")
print(f"% of cohort:            {len(cohort_schiz)/len(cohort)*100:.1f}%")

# Save
cohort_schiz.to_csv(r"C:\Users\laiar\Desktop\TFG\data_sintetica\Data-sintetica\src\csv merged\cohort_schizophrenia_sociodemografica.csv", index=False)
print("CSV saved!")

Rows with schizophrenia diagnosis: 11687
Unique patients: 10120

F20 subtypes found:
 PR_COD_PS                        PS_DES
 C01-F20.0       ESQUIZOFRÈNIA PARANOIDE
 C01-F20.1  ESQUIZOFRÈNIA DESORGANITZADA
 C01-F20.2      ESQUIZOFRÈNIA CATATÒNICA
 C01-F20.3  ESQUIZOFRÈNIA INDIFERENCIADA
 C01-F20.5        ESQUIZOFRÈNIA RESIDUAL
C01-F20.81    TRASTORN ESQUIZOFRENIFORME
C01-F20.89  ALTRES TIPUS D'ESQUIZOFRÈNIA
 C01-F20.9 ESQUIZOFRÈNIA NO ESPECIFICADA
Total cohort:           473812
Schizophrenia patients: 9969
% of cohort:            2.1%
CSV saved!


## 3. Fàrmacs Dispensats

In [1]:
import pandas as pd
import numpy as np
import re

# Codis ATC d'antipsicòtics
ANTIPSICIOTICS_ATC = (
    'N05AD01', 'N05AD1',   # Haloperidol
    'N05AH02',              # Clozapina
    'N05AH03',              # Olanzapina
    'N05AH04',              # Quetiapina
    'N05AH06',              # Clotiapina
    'N05AL01',              # Sulpirida
    'N05AL05',              # Amisulpirida
    'N05AX08',              # Risperidona
    'N05AX12',              # Aripiprazol
    'N05AX13',              # Paliperidona
    'N05AE05',              # Lurasidona
)

def es_antipsicotic(serie_atc: pd.Series) -> pd.Series:
    """Retorna màscara booleana: True si el codi ATC és un antipsicòtic."""
    return serie_atc.str.startswith(ANTIPSICIOTICS_ATC, na=False)

#  Funció per llegir en chunks i filtrar al vol 
def llegir_filtrant(path: str, cols: list[str], chunk_size: int = 50_000) -> pd.DataFrame:
    """
    Llegeix un CSV gran en trossos (chunks) i només guarda
    les files d'antipsicòtics. Molt més eficient en memòria.
    """
    chunks_filtrats = []
    
    for chunk in pd.read_csv(
        path,
        sep=',',
        encoding='latin1',
        usecols=cols,           # <-- llegeix NOMÉS les columnes que necessites
        chunksize=chunk_size,
        low_memory=False
    ):
        mask = es_antipsicotic(chunk['ID_HIS_Subgrup_7_ATC'])
        chunks_filtrats.append(chunk[mask])
    
    return pd.concat(chunks_filtrats, ignore_index=True)

### OF (Oficina de farmàcia)
23_farmacs_dispensants_2010_2019_Final

In [18]:
import pandas as pd

path_farmacs_dispensats = r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\23_farmacs_dispensants_2010_2019_Final.csv"

cols = pd.read_csv(path_farmacs_dispensats, sep=',', encoding='latin1', nrows=0).columns.tolist()
print(cols)

['id', 'idCas', 'Any_Mes', 'ID_HIS_Subgrup_7_ATC', 'DES_HIS_Subgrup_7_ATC', 'ID_HIS_Producte', 'DES_HIS_Producte', 'HIS_Quantitat_PA_Espec', 'ID_HIS_Unitat_Mesura_PA_Espec', 'DES_HIS_Unitat_Mesura_PA_Espec', 'HIS_Unitats_Productes', 'ID_HIS_Forma_Farmaceutica', 'DES_HIS_Forma_Farmaceutica', 'ID_HIS_Via_Administracio_Producte', 'DES_HIS_Via_Administracio_Producte', 'Nombre_Receptes_Dispensades', 'Nombre_DDD_Receptes_Dispensades', 'Import_Integre_Dispensades', 'Import_Liquid_Dispensat', 'Nombre_Envasos_Dispensats']


In [7]:
ANTIPSICIOTICS_ATC = (
    'N05AD01', 'N05AD1',   # Haloperidol
    'N05AH02',              # Clozapina
    'N05AH03',              # Olanzapina
    'N05AH04',              # Quetiapina
    'N05AH06',              # Clotiapina
    'N05AL01',              # Sulpirida
    'N05AL05',              # Amisulpirida
    'N05AX08',              # Risperidona
    'N05AX12',              # Aripiprazol
    'N05AX13',              # Paliperidona
    'N05AE05',              # Lurasidona
)

BASE = r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets"

def llegir_filtrant(path):
    chunks = []
    for chunk in pd.read_csv(path, sep=',', encoding='latin1', chunksize=50_000, low_memory=False):
        mask = chunk['ID_HIS_Subgrup_7_ATC'].str.startswith(ANTIPSICIOTICS_ATC, na=False)
        chunks.append(chunk[mask])
    return pd.concat(chunks, ignore_index=True)

df_dispensats = llegir_filtrant(f"{BASE}\\23_farmacs_dispensants_2010_2019_Final.csv")

print(df_dispensats.head())

   id     idCas  Any_Mes ID_HIS_Subgrup_7_ATC DES_HIS_Subgrup_7_ATC  \
0   3  68121038   201912              N05AX12           Aripiprazol   
1   7  68121038   201912              N05AX12           Aripiprazol   
2  12  69210559   201912              N05AX13          Paliperidona   
3  13  69210559   201911              N05AH04            Quetiapina   
4  14  69210559   201911              N05AX13          Paliperidona   

   ID_HIS_Producte                                   DES_HIS_Producte  \
0           728196                        ABILIFY 10MG 28 COMPRIMIDOS   
1           728196                        ABILIFY 10MG 28 COMPRIMIDOS   
2           723799  INVEGA 3MG 28 COMPRIMIDOS DE LIBERACION PROLON...   
3           661762  QUETIAPINA STADA 100MG 60 COM RE P BLIS PVC/AI...   
4           723799  INVEGA 3MG 28 COMPRIMIDOS DE LIBERACION PROLON...   

  HIS_Quantitat_PA_Espec ID_HIS_Unitat_Mesura_PA_Espec  \
0                     10                             4   
1                 

In [12]:
df_farmacs_dispensats=pd.read_csv(r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\23_farmacs_dispensants_2010_2019_Final.csv", sep=',', encoding='latin1')

C:\Users\LRAJAG\AppData\Local\Temp\ipykernel_20412\4087764429.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_farmacs_dispensats=pd.read_csv(r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\23_farmacs_dispensants_2010_2019_Final.csv", sep=',', encoding='latin1')


In [14]:
total_files = sum(len(chunk) for chunk in pd.read_csv(path_farmacs_dispensats, sep=',', encoding='latin1', chunksize=50_000, low_memory=False, usecols=['idCas']))

print(f"Farmacs Dispensats Total:    {total_files:,}")
print(f"Antipsicotics Dispensats Total: {len(df_dispensats):,}")

Farmacs Dispensats Total:    67,086,050
Antipsicotics Dispensats Total: 8,140,469


In [15]:
df_dispensats.to_csv(r'C:\Users\LRAJAG\Desktop\Data_sintetica\src\ADHERENCIA\antipsicotics_dispensats.csv', index=False)

### MHDA (Hospital)
Seleccionar mateixes columnes que farmacs OF però afegim pas de treure pacients amb idCas dins de cohort_control.
Pel futur merge amb farmacs_dispensats afegim columna d'envasos que és majoritàriament 1.

In [17]:
import pandas as pd

path_mhda = r"C:\Users\LRAJAG\Desktop\DATOS_PADRIS_nets\25_taula_farmacs_mhda_2010_2019_Final.csv"

cols = pd.read_csv(path_farmacs_dispensats, sep=',', encoding='latin1', nrows=0).columns.tolist()
print(cols)

['id', 'idCas', 'Any_Mes', '(HIS) Subgrup 7 (ATC)', '(HIS) Subgrup 7 (ATC) Desc', '(HIS) Forma FarmacÃ¨utica', '(HIS) Forma FarmacÃ¨utica Desc', '(HIS) Producte', '(HIS) Producte Desc', '(HIS) Via de distribuciÃ³', 'Concepte FacturaciÃ³', 'Concepte FacturaciÃ³ Desc', 'Import activitat MHDA']


In [20]:
total_files_mhda = 0
chunks_filtrats_mhda = []

for chunk in pd.read_csv(path_mhda, sep=',', encoding='latin1', chunksize=50_000, low_memory=False):
    total_files_mhda += len(chunk)
    mask = chunk['(HIS) Subgrup 7 (ATC)'].str.startswith(ANTIPSICIOTICS_ATC, na=False)
    chunks_filtrats_mhda.append(chunk[mask])

df_mhda = pd.concat(chunks_filtrats_mhda, ignore_index=True)

print(f"MHDA Total:          {total_files_mhda:,}")
print(f"MHDA Antipsicotics:  {len(df_mhda):,}")

MHDA Total:          837,826
MHDA Antipsicotics:  2,580


In [21]:
df_mhda.to_csv(r'C:\Users\LRAJAG\Desktop\Data_sintetica\src\ADHERENCIA\antipsicotics_mhda.csv', index=False)